# Scaling up: $N$ Coupled Classical Oscillators via `paulikit`

This notebook is a companion to
[`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb),
which develops the theory (state encoding, Pauli decomposition,
Suzuki-Trotter evolution, Trotter-error bounds, the Hadamard test) on
the $N=2$ case. Section references written "N=2 §x.y" point there.

The $N=2$ notebook builds the Hamiltonian symbolically in SymPy and
Pauli-decomposes it by brute force - for each of the $4^n$ Pauli
labels it builds the $2^n\times2^n$ tensor product and a full matrix
product before taking the trace, $O(32^n)$ in total. The notebook then 
runs the circuit through Classiq's cloud `synthesize`/`execute`. Here, 
we redo the Hamiltonian, decomposition, and evolution steps without 
either dependency:

1. **The Hamiltonian** is built numerically via
   [`paulikit`](https://pypi.org/project/paulikit/)'s
   `build_hamiltonian` instead of SymPy - SymPy's own symbolic engine
   is part of what caps $N\approx4$, independent of how the
   decomposition is done.
2. **The decomposition** uses `paulikit`'s Fast Walsh-Hadamard
   Transform ($O(n\cdot4^n)$) instead of the brute-force trace loop.
3. **The Trotter-repetition bound** (`alpha_comm`, Proposition 16 of
   Childs, Su, Tran, Wiebe, Zhu, "Theory of Trotter Error with
   Commutator Scaling", PRX 11, 011020 (2021)) is replaced with a
   coefficient-only upper
   bound (a closed form, tightened by which Pauli strings
   anticommute) - the exact version needs a
   per-term operator-2-norm (an SVD) that does not scale past a few
   hundred Pauli terms, let alone the thousands of the default
   $N=30$.
4. **The Trotterized evolution itself** is run with the
   repository's `trotterkit` statevector simulator (§5 below)
   instead of Classiq's cloud `synthesize`/`execute` - a real
   engineering decision, not a drop-in substitute, made explicit
   there.
5. **The Hadamard test** that recovers signs and phases from
   measurement outcomes alone is built from our own pieces (§8): a
   controlled evolution run through `trotterkit`, emulated
   measurement with shot noise, and an exact gate count - again no
   Classiq.

What the notebook itself verifies: `paulikit`'s decomposition
round-trips on the $N=2$ case; at the chosen $N$ the whole Hamiltonian
matrix is compared against an independent construction from the
physics (§2), every Pauli coefficient is recomputed by direct
projection and checked by a Parseval completeness test (§2); the
Trotter bound is checked in §3 against the exact Proposition 16 value
at $N=2$ (it must be an upper bound there); the Trotterized evolution
is compared with exact evolution in §5, and the positions and
velocities decoded from the evolved state (§7) are compared with the
exact classical solution; the controlled evolution of §8 is checked
branch by branch against the plain evolution, and the emulated shot
noise is checked to fall as $1/\sqrt{\text{shots}}$.

**Choosing $N$.** The number of oscillators is the single parameter
`N` in the cell below (any integer $N\ge2$; default $N=30$); every
other cell derives from it. The cost grows quickly with $N$ (more
qubits, more Pauli terms, and more Trotter steps from the §3 bound),
so a cost-preview cell (§4.1) times a few evolution steps at the
chosen $N$ and prints the estimated run time before any long cell
starts. If the estimate exceeds about 15 minutes it stops with an
error unless `ALLOW_LONG_RUN = True`.

For the default $N=30$, running the notebook top to bottom takes about
3 to 4 minutes on an Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, nearly
all of it the 15,493-step trajectory of §7 (about 2.5 to 3 minutes);
§8 adds about 20 seconds. Numbers quoted in the text "for the default
$N=30$" are from that run; for any other $N$ the printed outputs are
the reference.

## Prerequisites

Before working through this notebook, you should be familiar with:

- The $N=2$ notebook, in particular its §4 (Pauli decomposition),
  §5 (Suzuki-Trotter decomposition) and §6.6.1 (Trotter-error bounds)
- The Pauli group: $P^2=I$, and that every Pauli string is unitary
- Basic NumPy and SciPy sparse matrices

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build and Pauli-decompose an $N$-oscillator Hamiltonian numerically
   with `paulikit`, and verify every coefficient by direct projection.
2. Derive a coefficient-only upper bound on the 2nd-order Trotter
   error and turn it into a repetition count.
3. Apply a Pauli-string exponential to a statevector in $O(\dim)$,
   without building any $2^n\times2^n$ matrix.
4. Run the Trotterized evolution with `trotterkit`, verified against
   exact evolution, with a progress tracker on the long loop.
5. Decode the physical positions and velocities from the evolved
   state, including the free-end centre-of-mass correction, and
   validate them against the exact classical solution.
6. Recover amplitudes with signs and phases from measurement outcomes
   alone with a Hadamard test, implement its controlled evolution
   exactly with `trotterkit`, and count the circuit's gates.

## File Paths and Working Directory

This notebook can be run from any directory. It needs the
`trotterkit` package, which `./bootstrap` (or `make trotterkit`)
builds into the environment from the repository root.

## Parameters

`N` is the number of oscillators (an integer $\ge2$). The default
$N=30$ is what the module's continuous-integration run executes.
`ALLOW_LONG_RUN` lets the notebook continue when the cost preview of
§4.1 estimates more than about 15 minutes. The cell is tagged
`parameters`, so a tool such as papermill can set both from outside.

In [ ]:
N = 30                  # number of oscillators, any integer >= 2
ALLOW_LONG_RUN = False  # True: run even if the cost preview says > ~15 min

## 1. Physical setup: $N$ coupled oscillators

Two conventions are offered, both producing a valid
$(N+M)\times(N+M)$ Hamiltonian via the same `build_hamiltonian` call:

- **Deterministic, nearest-neighbor coupling** (default below): unit
  mass at every oscillator, unit spring constant between adjacent
  oscillators only, zero elsewhere - the simplest 1D chain, easy to
  reason about by hand. It has no wall springs (every on-site
  $k_{ii}=0$), so its ends are free and the stiffness matrix has a
  zero mode: the centre of mass moves uniformly. Positions are
  recovered later (§7) from the spring stretches plus the
  centre-of-mass motion.
- **Seeded random couplings**: every allowed $(i,j)$ pair (on-site and
  coupling) gets an independent random spring constant, masses randomized
  too - exercises the pipeline on a generic coupling network rather
  than one special case, with a fixed seed so results are
  reproducible run to run. Full coupling is far more expensive to
  evolve. For the default $N=30$: $107{,}008$ Pauli terms instead of
  $14{,}080$, and the
  §3 bound asks for $8{,}726$ repetitions at $t=1.0$
  instead of $439$ - about 11 minutes for §6 with `trotterkit`
  (about 73 ms per repetition, measured on this machine). Every
  check through §5 still runs in seconds, except §5's check, which
  needs about 4 minutes here; for §6, set `t_demo = 0.1` in §4.1 ($276$
  repetitions, about 20 seconds). §7's full window would need
  $308{,}105$ repetitions, about 6 hours; set `t_end = 0.5` in §4.1
  ($3{,}086$ repetitions, about 4.5 minutes).

Both are built as a `scipy.sparse` matrix (`sparse=True`). The
matrix has $N+N(N+1)/2$ rows, padded to the next power of two; the
default $N=30$ pads to $9$ qubits ($\dim=512$; the size comes from
`paulikit`'s layout of one $B$ column per $(i,j)$ pair, not from the
physics - see §2.1), small enough that dense would also work
there, but sparse construction is what lets this same code reach far
larger $N$ without a rewrite: the `paulikit` paper
([arXiv:2610.08299](https://arxiv.org/abs/2610.08299)) runs this same
coupled-oscillator family up to $1{,}470{,}038{,}016$ surviving
terms, with every coefficient checked by exhaustive projection up to
the $91{,}652{,}096$-term $N=150$ case.

In [ ]:
####################################################
# Physical parameters for N oscillators:
# deterministic or random
#  - build_nearest_neighbor_params(n)
#  - build_random_params(n, seed)
####################################################

import numbers

import numpy as np

# Validate the parameter cell before any heavy work.
if (not isinstance(N, numbers.Integral) or isinstance(N, bool)
        or N < 2):
    raise ValueError(f"N must be an integer >= 2, got {N!r}")
N = int(N)
if not isinstance(ALLOW_LONG_RUN, bool):
    raise ValueError("ALLOW_LONG_RUN must be True or False, got "
                     f"{ALLOW_LONG_RUN!r}")


def build_nearest_neighbor_params(n):
    """Unit mass, unit nearest-neighbor spring constant, zero
    elsewhere - the simplest standard 1D coupled-oscillator chain.
    Every (i, j) pair with i <= j must be present (build_hamiltonian
    requires it), so non-adjacent pairs are set to 0.0 explicitly.
    """
    masses = [1.0] * n
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = 1.0 if jj == i + 1 else 0.0
    return masses, spring_constants


def build_random_params(n, seed):
    """Independent random spring constants for every allowed (i, j)
    pair and random positive masses, seeded for reproducibility.
    Spring constants are drawn as abs(normal) so the Hamiltonian stays
    a physically sensible (non-negative stiffness) coupling network.
    """
    rng = np.random.default_rng(seed)
    masses = (1.0 + rng.random(n)).tolist()
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = float(abs(rng.normal(loc=1.0, scale=0.3)))
    return masses, spring_constants


# Choice of convention for the rest of this section - switch to
# build_random_params(N, seed=0) to exercise the random case instead.
masses, spring_constants = build_nearest_neighbor_params(N)
print(f"N={N}: {len(spring_constants)} (i,j) spring-constant entries, "
      f"{sum(1 for v in spring_constants.values() if v != 0.0)} nonzero")

## 2. Hamiltonian and Pauli decomposition via `paulikit`

In [ ]:
####################################################
# Build the N-oscillator Hamiltonian, decompose it
#  - N=2: dense round-trip reconstruction
#  - chosen N: every term checked by direct projection
####################################################

from importlib.metadata import version as _pkg_version

from paulikit.hamiltonian import build_hamiltonian, pad_to_power_of_two
from paulikit.algorithms.fwht import fwht_pauli_terms
from paulikit.pauli_utils import reconstruct_from_terms

_pk_version = tuple(
    int(p) for p in _pkg_version("paulikit").split(".")[:3] if p.isdigit())
if _pk_version < (0, 1, 4):
    raise RuntimeError("paulikit >= 0.1.4 is required (earlier versions "
                       "build a wrong Hamiltonian for N >= 3), found "
                       + _pkg_version("paulikit"))

# --- Correctness check first, on the already-verified N=2 case,
# before trusting paulikit's decomposition at the chosen N ---
_H2 = build_hamiltonian(2, {(0, 0): 1.0, (0, 1): 2.0, (1, 1): 3.0},
                        [1.0, 2.0], sparse=False)
_H2p, _n2 = pad_to_power_of_two(_H2, sparse=False)
_terms2 = fwht_pauli_terms(_H2p, assume_hermitian=True)
_H2_rebuilt = reconstruct_from_terms(_terms2, _n2)
_max_err_n2 = np.max(np.abs(_H2_rebuilt - _H2p))
print(f"N=2 sanity check: {len(_terms2)} terms, "
      f"reconstruction max abs error = {_max_err_n2:.2e}")
if _max_err_n2 > 1e-10:
    raise RuntimeError("paulikit decomposition does not round-trip at N=2")

# --- Now the real N-oscillator Hamiltonian ---
H = build_hamiltonian(N, spring_constants, masses, sparse=True)
Hp, n_qubits = pad_to_power_of_two(H, sparse=True)
dim = 2 ** n_qubits
print(f"N={N}: padded dimension={dim}, n_qubits={n_qubits}")

# --- Physics check: the checks below only test the decomposition of
# whatever matrix paulikit returns, so they cannot catch a wrong
# Hamiltonian. H = -[[0, B], [B^T, 0]] reproduces Newton's equations
# only if B B^T = M^{-1/2} F M^{-1/2}, F the stiffness matrix
# (f_jj = sum_k k_jk, f_jk = -k_jk). paulikit < 0.1.4 fails this
# for N >= 3. ---
_F = np.zeros((N, N))
for (i, j), kappa in spring_constants.items():
    _F[i, i] += kappa
    if i != j:
        _F[j, j] += kappa
        _F[i, j] -= kappa
        _F[j, i] -= kappa
_inv_sqrt_m = np.diag(1.0 / np.sqrt(masses))
_H_dense = H.toarray()
_B = -_H_dense[:N, N:]
_phys_err = np.max(np.abs(_B @ _B.T - _inv_sqrt_m @ _F @ _inv_sqrt_m))
print(f"N={N} physics check: |B B^T - M^-1/2 F M^-1/2| = {_phys_err:.2e}")
if _phys_err > 1e-12:
    raise RuntimeError("Hamiltonian does not reproduce Newton's equations "
                       "(paulikit >= 0.1.4 required)")

# B rebuilt independently from the physics: one column per (i, j) pair
# with i <= j, in paulikit's column order. Wall spring i -> column i,
# with +sqrt(k_ii/m_i) in row i; couplings (j, k), j < k, in row-major
# order -> columns N, N+1, ..., with +sqrt(k_jk/m_j) in row j and
# -sqrt(k_jk/m_k) in row k. The WHOLE matrix is then compared, which
# also pins both diagonal blocks to zero and the lower-left to B^T.
_M = N * (N + 1) // 2
_B_phys = np.zeros((N, _M))
for _i in range(N):
    _B_phys[_i, _i] = np.sqrt(spring_constants[(_i, _i)] / masses[_i])
_col = N
for _j in range(N):
    for _k in range(_j + 1, N):
        _kap = spring_constants[(_j, _k)]
        _B_phys[_j, _col] = np.sqrt(_kap / masses[_j])
        _B_phys[_k, _col] = -np.sqrt(_kap / masses[_k])
        _col += 1
_H_phys = -np.block([[np.zeros((N, N)), _B_phys],
                     [_B_phys.T, np.zeros((_M, _M))]])
_whole_err = np.max(np.abs(_H_dense - _H_phys))
print(f"N={N} whole-matrix check: max|H - (-[[0,B],[B^T,0]])| = "
      f"{_whole_err:.2e}")
if _whole_err > 1e-12:
    raise RuntimeError("Hamiltonian differs from the one built "
                       "independently from the physics")

pauli_terms = fwht_pauli_terms(Hp, assume_hermitian=True)
pauli_labels = list(pauli_terms.keys())
pauli_coeffs = np.array([pauli_terms[label] for label in pauli_labels],
                           dtype=np.float64)
print(f"N={N}: {len(pauli_labels)} nonzero Pauli terms "
      f"(out of up to 4**{n_qubits} = {4 ** n_qubits} possible)")

# --- Correctness check at the chosen N ---
# Dense round-trip reconstruction (used above for N=2) builds one
# dim x dim matrix per term (~45 s at N=30), so instead every
# claimed coefficient is recomputed independently by direct
# projection, c_P = Tr(H P^dagger) / dim, using that a Pauli string
# has one nonzero per row (row r -> column r ^ x_mask). This checks
# all terms, not a sample. Projection only checks the terms paulikit
# reported; Parseval (dim * sum|c|^2 == ||H||_F^2) catches any term
# it dropped. Same method as paulikit's own
# verification/exhaustive_projection.py.


def labels_to_masks(labels, n_qubits):
    """Symplectic (x_mask, z_mask) per label; label[0] = qubit 0 = MSB."""
    chars = np.frombuffer("".join(labels).encode(), dtype=np.uint8)
    chars = chars.reshape(len(labels), n_qubits)
    weights = np.int64(1) << np.arange(n_qubits - 1, -1, -1, dtype=np.int64)
    x_bits = (chars == ord("X")) | (chars == ord("Y"))
    z_bits = (chars == ord("Z")) | (chars == ord("Y"))
    return x_bits @ weights, z_bits @ weights


def verify_by_projection(H, terms, n_qubits, chunk=None):
    """Return (max coefficient error, relative Parseval gap)."""
    H = H.toarray() if hasattr(H, "toarray") else np.asarray(H)
    dim = 2 ** n_qubits
    if chunk is None:
        # chunk x dim work arrays: at most ~4M entries (~32 MB each),
        # 4096 terms per chunk up to dim = 1024
        chunk = max(1, min(4096, 2 ** 22 // dim))
    labels = list(terms)
    claimed = np.array([terms[k] for k in labels], dtype=np.complex128)
    x, z = labels_to_masks(labels, n_qubits)
    r = np.arange(dim, dtype=np.int64)
    projected = np.empty(len(labels), dtype=np.complex128)
    for s in range(0, len(labels), chunk):
        xs, zs = x[s:s + chunk], z[s:s + chunk]
        cols = r[None, :] ^ xs[:, None]
        sign = 1 - 2 * (np.bitwise_count(r[None, :] & zs[:, None])
                        & 1).astype(np.int64)
        phase = 1j ** (np.bitwise_count(xs & zs) & 3)
        projected[s:s + chunk] = ((H[r[None, :], cols] * sign).sum(axis=1)
                                  * phase / dim)
    frob2 = np.sum(np.abs(H) ** 2)
    parseval_gap = abs(dim * np.sum(np.abs(claimed) ** 2) - frob2) / frob2
    return np.max(np.abs(projected - claimed)), parseval_gap


_proj_err, _parseval_gap = verify_by_projection(
    Hp, pauli_terms, n_qubits)
print(f"N={N} projection check, all {len(pauli_terms)} terms: "
      f"max abs error = {_proj_err:.2e}, "
      f"relative Parseval gap = {_parseval_gap:.2e}")
if _proj_err > 1e-10 or _parseval_gap > 1e-12:
    raise RuntimeError(f"paulikit decomposition fails verification at N={N}")

### 2.1 Aside: dropping the all-zero columns of $B$

`paulikit` gives $B$ one column per pair $(i,j)$, $i\le j$, whether
or not that spring exists, so the nearest-neighbor chain carries
$N(N+1)/2$ columns of which only $N-1$ are nonzero. A zero column is a
stretch that never changes, so the components it would occupy stay
exactly zero under the evolution. The cell below checks that dropping
those columns changes nothing on the physical components, and times
one `trotterkit` step in both layouts. The rest of the notebook keeps
`paulikit`'s full layout; §8.2 counts the gates for both.

In [ ]:
####################################################
# Aside: drop the all-zero B columns, check exactness
#  - pruned H on N + (nonzero springs) components
####################################################

import time

try:
    import trotterkit
except ImportError as exc:
    raise ImportError(
        "trotterkit is not installed: run `make tools` (or "
        "./bootstrap) from the repository root, then restart "
        "this kernel") from exc
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import expm_multiply

_nz_cols = np.flatnonzero(np.any(_B_phys != 0.0, axis=0))
print(f"B has {_M} columns, {len(_nz_cols)} nonzero, "
      f"{_M - len(_nz_cols)} zero")
_Bp = _B_phys[:, _nz_cols]
_H_pruned_small = csr_matrix(-np.block([
    [np.zeros((N, N)), _Bp],
    [_Bp.T, np.zeros((len(_nz_cols), len(_nz_cols)))]]))
_H_pruned, _n_q_pruned = pad_to_power_of_two(_H_pruned_small, sparse=True)
_terms_pruned = fwht_pauli_terms(_H_pruned, assume_hermitian=True)
print(f"pruned H: {N + len(_nz_cols)} components -> "
      f"{2 ** _n_q_pruned} ({_n_q_pruned} qubits), "
      f"{len(_terms_pruned)} Pauli terms "
      f"(full layout: {n_qubits} qubits, {len(pauli_terms)} terms)")

# Same initial state in both layouts: physical components only.
_psi_full = np.zeros(dim, dtype=np.complex128)
_psi_full[0], _psi_full[1] = 1.0 / np.sqrt(2), -1.0 / np.sqrt(2)
_psi_pruned = np.zeros(2 ** _n_q_pruned, dtype=np.complex128)
_psi_pruned[0], _psi_pruned[1] = _psi_full[0], _psi_full[1]

_t_aside = 1.0
_out_full = expm_multiply(-1j * _t_aside * Hp, _psi_full)
_out_pruned = expm_multiply(-1j * _t_aside * _H_pruned, _psi_pruned)
_keep = np.concatenate([np.arange(N), N + _nz_cols])
_aside_err = np.max(np.abs(_out_full[_keep]
                           - _out_pruned[:len(_keep)]))
_dropped = np.setdiff1d(np.arange(dim), _keep)
_dropped_max = np.max(np.abs(_out_full[_dropped]))
print(f"exact evolution at t={_t_aside}: max|full - pruned| on the "
      f"{len(_keep)} physical components = {_aside_err:.2e}; largest "
      f"dropped (zero-spring/padding) component = {_dropped_max:.2e}")
if _aside_err > 1e-12 or _dropped_max > 1e-12:
    raise RuntimeError("dropping the zero springs is not exact")


def _time_step(psi, x, z, coeffs, budget_s=0.3, max_reps=128):
    """Seconds per 2nd-order trotterkit step (dt is arbitrary). After
    one warm-up step, runs 1, 2, 4, ... steps in one call until a
    call takes budget_s or reaches max_reps steps, so the per-call
    overhead is amortised as in a long run, and the probe stays
    short at large N."""
    trotterkit.evolve(psi, t=0.01, steps=1, terms=(x, z, coeffs),
                      order=2, backend="c")
    reps = 1
    while True:
        start = time.perf_counter()
        trotterkit.evolve(psi, t=0.01 * reps, steps=reps,
                          terms=(x, z, coeffs), order=2, backend="c")
        elapsed = time.perf_counter() - start
        if elapsed >= budget_s or reps >= max_reps:
            return elapsed / reps
        reps *= 2


_xf, _zf = labels_to_masks(pauli_labels, n_qubits)
_xp, _zp = labels_to_masks(list(_terms_pruned), _n_q_pruned)
_cp = np.array([_terms_pruned[k].real for k in _terms_pruned])
_step_full = _time_step(
    _psi_full, np.ascontiguousarray(_xf, dtype=np.uint64),
    np.ascontiguousarray(_zf, dtype=np.uint64), pauli_coeffs)
_step_pruned = _time_step(
    _psi_pruned, np.ascontiguousarray(_xp, dtype=np.uint64),
    np.ascontiguousarray(_zp, dtype=np.uint64), _cp)
print(f"one trotterkit step: full {1e3 * _step_full:.2f} ms, pruned "
      f"{1e3 * _step_pruned:.2f} ms ({_step_full / _step_pruned:.1f}x)")

## 3. Trotter-repetition bound: closed-form, coefficient-only

<div style="border: 2px solid orange; padding: 10px;">

This bound is a **provably valid but looser** version of the exact
Proposition 16 bound used in N=2 §6.6.1. The exact bound needs the
operator 2-norm of two nested commutators for every Pauli term. At
$N=2$ that is 12 terms and $8\times8$ matrices; at the default
$N=30$ it is $14{,}080$ terms, each needing nested commutators of
dense $512\times512$ matrices (a few matrix products each, plus a top
singular value). A single such norm is cheap; the obstacle is the
count - about $14{,}000$ of them, each on a tail matrix that must
itself be re-formed for every term - and it grows with the term
count (the random case at $N=30$ has $107{,}008$) and with $N$. The bounds below need no
matrices at all - only the Pauli coefficients and, for the tighter
one, which pairs of strings anticommute - but will generally
recommend *more* repetitions than the exact bound would for the same
target accuracy (never fewer: a looser upper bound on the true error
can only be conservative, not optimistic).

</div>

In [ ]:
####################################################
# Coefficient-only alpha_comm bounds
#  - compute_alpha_comm_2nd_order_closed_form(coeffs)
#  - compute_alpha_comm_2nd_order_anticommuting(x, z, coeffs)
#  - required_trotter_repetitions(t, target_error)
####################################################

import functools
import math
import time


def compute_alpha_comm_2nd_order_closed_form(coeffs):
    """Upper bound on Proposition 16's alpha_comm using ONLY the
    Pauli coefficients, no matrices and no operator norms at all.

    Starting point, exactly as in the N=2 notebook's Section 6.6.1:
      alpha_comm = (1/12) sum_g ||[tail_g, [tail_g, H_g]]||
                  + (1/24) sum_g ||[H_g, [H_g, tail_g]]||
    where H_g = c_g * P_g for Pauli string P_g (||P_g|| = 1, since
    every Pauli string is unitary) and tail_g = sum_{g'>g} H_g'.

    Two standard inequalities replace every operator norm with a sum
    over |c_g| alone:
      ||[A, B]|| <= 2 ||A|| ||B||           (sub-multiplicativity)
      ||tail_g|| <= sum_{g'>g} |c_g'|        (triangle inequality, ||P||=1)
    Applying the commutator bound twice (nested commutator) and the
    tail bound once per factor gives, per term g:
      ||[tail_g,[tail_g,H_g]]|| <= 4 * tail_g^2 * |c_g|
      ||[H_g,[H_g,tail_g]]||    <= 4 * |c_g|^2 * tail_g
    Summed over g, this is O(terms) via one running suffix-sum of
    |c_g| - no per-term matrix ever gets built.

    This is looser than the exact bound: it treats every pair of
    terms as if it anticommuted. On the N=2 notebook's 12 terms, in
    paulikit's term order, it gives about 4.857 against the exact
    0.5714, roughly 8.5x looser.
    """
    coeffs = np.asarray(coeffs, dtype=float)
    abs_c = np.abs(coeffs)
    n = len(abs_c)
    # suffix_sum[g] = sum_{g' > g} |c_g'|
    suffix_sum = np.zeros(n)
    if n > 1:
        suffix_sum[:-1] = np.cumsum(abs_c[::-1])[::-1][1:]
    term1 = np.sum(4.0 * suffix_sum ** 2 * abs_c)
    term2 = np.sum(4.0 * abs_c ** 2 * suffix_sum)
    return term1 / 12.0 + term2 / 24.0


def compute_alpha_comm_2nd_order_anticommuting(
        x, z, coeffs, block=1024, col_block=4096, report_every_s=10.0):
    """Tighter coefficient-only upper bound on alpha_comm, which also
    uses which pairs of Pauli strings anticommute.

    Two Pauli strings either commute, [P_a, P_b] = 0, or anticommute,
    ||[P_a, P_b]|| = 2. So, with
      A_g = sum over g' > g with P_g' ANTICOMMUTING with P_g of |c_g'|
    (and S_g = sum_{g'>g} |c_g'| as before), ||[tail_g, H_g]|| is at
    most 2 |c_g| A_g, and
      ||[tail_g,[tail_g,H_g]]|| <= 2 ||tail_g|| ||[tail_g,H_g]||
                                <= 4 S_g A_g |c_g|
      ||[H_g,[H_g,tail_g]]||    <= 2 |c_g| ||[H_g,tail_g]||
                                <= 4 c_g^2 A_g
    Since A_g <= S_g this is never looser than the closed form above.

    Strings (x_a, z_a) and (x_b, z_b) (bit masks) anticommute iff
    popcount((x_a & z_b) ^ (z_a & x_b)) is odd. All pairs are tested
    in numpy blocks of block x col_block, so memory stays small
    (about 100 MB); the work is about T^2 / 2 pair tests. Measured on
    this machine at N=30: about 1 s for the chain's 14,080 terms and
    about 40 s for the random case's 107,008 terms. A progress line is
    printed if it runs longer than report_every_s.
    """
    abs_c = np.abs(np.asarray(coeffs, dtype=float))
    n = len(abs_c)
    x = np.asarray(x, dtype=np.uint64)
    z = np.asarray(z, dtype=np.uint64)
    anti_sum = np.zeros(n)
    start = time.perf_counter()
    last_report = start
    for s in range(0, n, block):
        e = min(s + block, n)
        xa, za = x[s:e, None], z[s:e, None]
        rows = np.arange(s, e)[:, None]
        for c in range(s, n, col_block):  # columns before s: g' <= g
            f = min(c + col_block, n)
            odd = np.bitwise_count((xa & z[None, c:f])
                                   ^ (za & x[None, c:f])) & 1
            odd = odd.astype(bool) & (np.arange(c, f)[None, :] > rows)
            anti_sum[s:e] += odd @ abs_c[c:f]
        now = time.perf_counter()
        if now - last_report >= report_every_s:
            done = (e * (2 * n - e)) / (n * n)  # share of pairs done
            eta = (now - start) * (1 - done) / done
            print(f"anticommuting bound: {100 * done:.0f}% of pair "
                  f"tests | elapsed {now - start:.0f}s | "
                  f"ETA {eta:.0f}s   ", end="\r", flush=True)
            last_report = now
    suffix_sum = np.zeros(n)
    if n > 1:
        suffix_sum[:-1] = np.cumsum(abs_c[::-1])[::-1][1:]
    term1 = np.sum(4.0 * suffix_sum * anti_sum * abs_c)
    term2 = np.sum(4.0 * abs_c ** 2 * anti_sum)
    return term1 / 12.0 + term2 / 24.0


# --- Rigour check at N=2 (12 terms, paulikit's term order): the exact
# Proposition 16 alpha, from dense matrices and spectral norms, must
# not exceed either bound, and the tighter bound must not exceed the
# closed form ---
_PAULI_2x2 = {"I": np.eye(2), "X": np.array([[0, 1], [1, 0]]),
              "Y": np.array([[0, -1j], [1j, 0]]),
              "Z": np.diag([1.0, -1.0])}
_labels2 = list(_terms2)
_coeffs2 = np.array([_terms2[k].real for k in _labels2])
_mats2 = [functools.reduce(np.kron, [_PAULI_2x2[ch] for ch in k])
          for k in _labels2]


def _comm(a, b):
    return a @ b - b @ a


_alpha_exact_2 = 0.0
for _g in range(len(_labels2)):
    _Hg = _coeffs2[_g] * _mats2[_g]
    _tail = sum((_coeffs2[b] * _mats2[b]
                 for b in range(_g + 1, len(_labels2))),
                np.zeros((8, 8), dtype=complex))
    _alpha_exact_2 += (
        np.linalg.norm(_comm(_tail, _comm(_tail, _Hg)), 2) / 12.0
        + np.linalg.norm(_comm(_Hg, _comm(_Hg, _tail)), 2) / 24.0)
_x2, _z2 = labels_to_masks(_labels2, _n2)
_alpha_closed_2 = compute_alpha_comm_2nd_order_closed_form(_coeffs2)
_alpha_anti_2 = compute_alpha_comm_2nd_order_anticommuting(
    _x2, _z2, _coeffs2)
print(f"N=2 alpha_comm: exact {_alpha_exact_2:.4f} <= anticommuting "
      f"{_alpha_anti_2:.4f} <= closed-form {_alpha_closed_2:.4f}")
if not (_alpha_exact_2 <= _alpha_anti_2 + 1e-12
        and _alpha_anti_2 <= _alpha_closed_2 + 1e-12):
    raise RuntimeError("Trotter bound ordering exact <= tighter <= "
                       "closed-form violated at N=2")

# --- The bounds at the chosen N ---
_x_bounds, _z_bounds = labels_to_masks(pauli_labels,
                                            n_qubits)
_alpha_closed = compute_alpha_comm_2nd_order_closed_form(
    pauli_coeffs)
_t_alpha = time.perf_counter()
_alpha = compute_alpha_comm_2nd_order_anticommuting(
    _x_bounds, _z_bounds, pauli_coeffs)
print(f"alpha_comm, N={N}, {len(pauli_coeffs)} terms: "
      f"closed-form {_alpha_closed:.6g}, "
      f"anticommuting {_alpha:.6g} "
      f"({_alpha_closed / _alpha:.2f}x tighter, "
      f"{time.perf_counter() - _t_alpha:.1f}s)" + " " * 20)


def required_trotter_repetitions(t, target_error, alpha=_alpha):
    """Repetitions r such that the Proposition-16 bound with the
    anticommuting alpha is <= target_error: alpha * t^3 / r^2 <=
    target_error."""
    if t == 0:
        return 1
    r = math.sqrt(alpha * abs(t) ** 3 / target_error)
    return max(1, math.ceil(r))


for _t in (0.1, 1.0, 5.0, 10.762932):
    print(
        f"t={_t:9.4f}  repetitions needed for target_error=0.05: "
        f"{required_trotter_repetitions(_t, 0.05, _alpha)} "
        f"(closed-form: "
        f"{required_trotter_repetitions(_t, 0.05, _alpha_closed)})"
    )

## 4. Choosing a demonstration time window

The $N=2$ notebook evolves to $t\approx10.76$. For the default
$N=30$ that needs $15{,}493$ repetitions under the (looser) §3 bound
(the plain closed form would ask for $22{,}058$) - correct, and
runnable (about 3 minutes with `trotterkit` below: about 11 ms per
repetition, measured on this machine, with a measured 2-norm error of
$4.9\times10^{-7}$ against exact evolution, far below the $0.05$
target), but slow enough to be a poor default for the §6
demonstration run. $t=1.0$ needs only $439$ repetitions at $N=30$
(about 4 seconds) and is used there; setting `t_demo = 10.762932` in
the next cell reproduces the longer run. §7 does run the full window,
once, to read out the trajectories. This is a choice of
**demonstration window, not a change in correctness** - the same
Hamiltonian, decomposition, and bound apply at either time. For other
$N$ the repetition counts printed by §3 and by the cost preview below
apply.

### 4.1 Cost preview

The repetition count grows with $N$ through the bound's $\alpha$, and
each repetition costs more (more terms on a longer statevector). The
next cell fixes the time windows and the error target, then *measures*
one Trotter step at this $N$ - the uncontrolled step of §5-7 (timed in
§2.1) and the controlled step of the §8 Hadamard test (timed here,
with a few steps) - plus one decode of §7, and multiplies by the step
counts each later cell will run. The dense exact-reference checks
(SciPy's `expm_multiply` and one eigendecomposition) are not in the
estimate; they take seconds at the default $N$. If the total exceeds
about 15 minutes the cell stops with an error; set
`ALLOW_LONG_RUN = True` in the parameter cell to run anyway.

In [ ]:
####################################################
# Cost preview: time a few steps at this N, estimate
# the later cells, stop if too long
#  - controlled_terms(x, z, coeff, n_qubits)
####################################################

t_demo = 1.0  # §6 window; set to 10.762932 to match the N=2
              # walkthrough's own time point instead (about 3 min for
              # the N=30 chain; the fully coupled random case at N=30
              # needs about 11 min at 1.0 and about 6 hours at
              # 10.762932)
t_end = 10.762932  # §7 trajectory window (the N=2 walkthrough's time)
target_error = 0.05  # 2-norm Trotter error target for the §3 bound
n_times = 41  # read-out times in §7 and §8
LONG_RUN_LIMIT_S = 15 * 60

# Same term order as pauli_labels / pauli_coeffs: the Trotter bound in
# section 3 was computed for exactly this order.
_x_m, _z_m = labels_to_masks(pauli_labels, n_qubits)
x_terms = np.ascontiguousarray(_x_m, dtype=np.uint64)
z_terms = np.ascontiguousarray(_z_m, dtype=np.uint64)
coeff_terms = pauli_coeffs


def controlled_terms(x, z, coeff, n_qubits):
    """Term list of the evolution controlled on an ancilla, the most
    significant of n_qubits + 1 qubits: each term (x, z, c) becomes
    the commuting pair (x, z, c/2), (x, z | bit_a, -c/2). Derived and
    verified in section 8."""
    bit_a = np.uint64(1) << np.uint64(n_qubits)
    x_c = np.repeat(x, 2)
    z_c = np.repeat(z, 2)
    z_c[1::2] |= bit_a
    c_c = np.repeat(np.asarray(coeff) / 2.0, 2)
    c_c[1::2] *= -1.0
    return x_c, z_c, c_c


def _mmss(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}:{seconds % 60:02d}"


# --- Timing probes at this N ---
_xc_probe, _zc_probe, _cc_probe = controlled_terms(
    x_terms, z_terms, coeff_terms, n_qubits)
_psi_probe = np.zeros(2 * dim, dtype=np.complex128)
_psi_probe[0] = 1.0
_step_ctl = _time_step(_psi_probe, _xc_probe, _zc_probe, _cc_probe)
del _xc_probe, _zc_probe, _cc_probe, _psi_probe
# One decode of section 7 (an SVD of B^T and a least-squares solve),
# which the section 8 shot emulation repeats many times.
_n_probe = 0
_t0 = time.perf_counter()
while _n_probe < 3 or (_n_probe < 50
                       and time.perf_counter() - _t0 < 0.2):
    if _n_probe == 1:  # the first call (warm-up) is not counted
        _t0 = time.perf_counter()
    np.linalg.svd(_B_phys.T, full_matrices=False)
    np.linalg.lstsq(_B_phys.T, np.ones(_B_phys.shape[1]), rcond=1e-10)
    _n_probe += 1
_decode_s = (time.perf_counter() - _t0) / (_n_probe - 1)

# --- Step counts each later cell runs ---
_r_check = required_trotter_repetitions(0.5, target_error)  # §5, §8
_r_demo = required_trotter_repetitions(t_demo, target_error)  # §6
_r_traj = required_trotter_repetitions(t_end, target_error)  # §7
_n_decodes = n_times * (1 + 100 + 9 * 8)  # §8.1: demo, 100 repeats,
                                           # 9 shot counts x 8 seeds
_est = {
    "§5 trotterkit check": _r_check * _step_full,
    "§6 demonstration run": _r_demo * _step_full,
    "§7 trajectory": _r_traj * _step_full,
    "§8 controlled-evolution check": _r_check * (_step_full
                                                  + 2 * _step_ctl),
    "§8.1 shot emulation": _n_decodes * _decode_s,
}
_total = sum(_est.values())
print(f"N={N}: {n_qubits} qubits for the evolution, {n_qubits + 1} "
      f"for the Hadamard test (with the ancilla); "
      f"{len(pauli_coeffs):,} Pauli terms "
      f"({2 * len(pauli_coeffs):,} controlled)")
print(f"Trotter steps: {_r_traj:,} for the §7 trajectory to "
      f"t={t_end:g}, {_r_demo:,} for §6 (t={t_demo:g}), {_r_check:,} "
      f"for the t=0.5 checks")
print(f"measured: {1e3 * _step_full:.2f} ms per step, "
      f"{1e3 * _step_ctl:.2f} ms per controlled step, "
      f"{1e3 * _decode_s:.2f} ms per decode")
for _name, _s in _est.items():
    print(f"  {_name:31s} ~ {_mmss(_s):>8s}")
print(f"  {'estimated total':31s} ~ {_mmss(_total):>8s} "
      f"(plus the dense exact-reference checks)")
if _total > LONG_RUN_LIMIT_S and not ALLOW_LONG_RUN:
    raise RuntimeError(
        f"At N={N} the rest of this notebook would take about "
        f"{_total / 60:.0f} minutes ({_total / 3600:.1f} hours), above the "
        f"{LONG_RUN_LIMIT_S // 60}-minute limit. Set ALLOW_LONG_RUN = "
        f"True in the parameter cell to run it anyway, or choose a "
        f"smaller N.")

## 5. Trotterized evolution with `trotterkit`

The $N=2$ notebook uses Classiq's cloud
`synthesize`/`execute` to run the Trotterized circuit. At the
scale of this notebook's $N$ it instead evolves the statevector classically with
the repository's `trotterkit` package - a deliberate engineering
choice, not a drop-in substitute for Classiq, made for two concrete
reasons:

1. **Avoiding a cloud round-trip dependency for this demonstration.**
   Classiq's own `synthesize`/`execute` needs network access and
   authentication; a local simulator runs fully offline and
   deterministically.
2. **Performance at this term count.** `trotterkit` applies each
   Pauli exponential to the statevector exactly as the circuit's gate
   would - the same per-term product formula, term by term, forward
   and then backward for the 2nd-order (Strang) step - in a native
   loop over the $2^n$ amplitudes. It uses its C backend by default,
   chosen from a measured comparison of its C, C++ and Rust
   backends; all backends give identical results. It is built into
   the environment by `./bootstrap` (or `make trotterkit`).

Each Pauli-string term $P$ is applied to the statevector via its
closed form $e^{-i\theta P}=\cos\theta\,I-i\sin\theta\,P$ (valid
because $P^2=I$), and $P\psi$ itself is applied
*without ever building the $2^n\times2^n$ matrix*: every Pauli string
has exactly one nonzero entry per row (a bit-flip permutation from any
$X$/$Y$ factors, a sign/phase from any $Y$/$Z$ factors), so $P\psi$
costs $O(\dim)$ instead of $O(\dim^2)$. The same
one-nonzero-per-row structure is what makes the projection check in
§2 cheap. `trotterkit` takes each term as the pair of bit masks
$(x, z)$ built in §2 plus its coefficient.

Before use, the cell below checks `trotterkit` against SciPy's exact
`expm_multiply` evolution at $t=0.5$, with the repetition count the
§3 bound asks for. It also defines `evolve_with_progress`, which runs
a long evolution in chunks and prints a live step counter, elapsed
time and ETA.

In [ ]:
####################################################
# trotterkit evolution, with a progress tracker,
# verified against exact evolution before use
#  - evolve_with_progress(psi, terms=..., dt=..., n_steps=...)
####################################################

import time

try:
    import trotterkit
except ImportError as exc:
    raise ImportError(
        "trotterkit is not installed: run `make tools` (or "
        "./bootstrap) from the repository root, then restart "
        "this kernel") from exc
from scipy.sparse.linalg import expm_multiply

# x_terms, z_terms, coeff_terms (section 3's term order) and _mmss
# are defined in the cost preview (4.1).


def evolve_with_progress(psi, *, terms, dt, n_steps, order=2,
                         backend="c", label="evolve",
                         report_every_s=2.0):
    """Advance psi by n_steps product-formula steps of size dt with
    trotterkit.evolve, in chunks, printing a live progress line.

    Every chunk is a run of steps of the same size dt, so the result
    is the same product formula as one n_steps call (and consecutive
    calls compose): chunking changes only when the progress line is
    printed. The first chunk is about 10 steps, to measure the rate;
    later chunks take about one second each. Returns the new state.
    """
    start = time.perf_counter()
    last_report = start
    done = 0
    chunk = min(10, n_steps)
    while done < n_steps:
        chunk = min(chunk, n_steps - done)
        psi = trotterkit.evolve(psi, t=chunk * dt, steps=chunk,
                                terms=terms, order=order,
                                backend=backend)
        done += chunk
        now = time.perf_counter()
        elapsed = now - start
        rate = done / elapsed
        chunk = max(1, int(rate))
        if done < n_steps and now - last_report >= report_every_s:
            eta = (n_steps - done) / rate
            print(f"{label}: step {done}/{n_steps} "
                  f"({100.0 * done / n_steps:.0f}%) | "
                  f"elapsed {_mmss(elapsed)} | ETA {_mmss(eta)}   ",
                  end="\r", flush=True)
            last_report = now
    print(f"{label}: {n_steps} steps in "
          f"{_mmss(time.perf_counter() - start)}" + " " * 30)
    return psi


# --- Verify trotterkit against exact evolution before using it ---
_t_check = 0.5
_target_check = 0.05
_r_check = required_trotter_repetitions(_t_check, _target_check)
_psi0_check = np.zeros(dim, dtype=np.complex128)
_psi0_check[0] = 1.0 / np.sqrt(2)
_psi0_check[1] = -1.0 / np.sqrt(2)
_psi_tk = evolve_with_progress(
    _psi0_check, terms=(x_terms, z_terms, coeff_terms), dt=_t_check / _r_check,
    n_steps=_r_check, label="verify")
_psi_exact = expm_multiply(-1j * _t_check * Hp, _psi0_check)
_err_check = np.linalg.norm(_psi_tk - _psi_exact)
print(f"trotterkit vs exact expm_multiply at t={_t_check}, "
      f"{_r_check} steps: 2-norm error = {_err_check:.2e} "
      f"(target {_target_check})")
if _err_check > _target_check:
    raise RuntimeError("trotterkit evolution exceeds the Trotter "
                       "error target at t=0.5")


## 6. Running the demonstration

In [ ]:
####################################################
# Run the Trotterized evolution at the chosen
# demonstration time window
####################################################

import time

# t_demo and target_error are set in the cost preview (4.1).
r_demo = required_trotter_repetitions(t_demo, target_error)
dt_demo = t_demo / r_demo
print(f"t={t_demo}, target_error={target_error}: {r_demo} Trotter repetitions")

psi0 = np.zeros(dim, dtype=np.complex128)
psi0[0] = 1.0 / np.sqrt(2)
psi0[1] = -1.0 / np.sqrt(2)

_t0 = time.perf_counter()
psi_demo = evolve_with_progress(
    psi0, terms=(x_terms, z_terms, coeff_terms), dt=dt_demo,
    n_steps=r_demo, label=f"N={N} evolution")
_t1 = time.perf_counter()

print(f"ran {r_demo} repetitions in {_t1 - _t0:.1f}s")
print(f"final state norm (should be 1.0): {np.linalg.norm(psi_demo):.10f}")


## 7. Reading out the trajectories

So far the evolved state has only been compared as a vector. This
section decodes the physical **positions and velocities** from it, at
41 times across $t\in[0, 10.762932]$, and validates them against the
exact classical solution.

**Encoding** (Babbush et al. 2023; N=2 §3). With $y=\sqrt{M}\,x$ the
state is $\psi=(\dot y,\; iB^{T}y)$ - the first $N$ components are the
scaled velocities, the next $P=N(N+1)/2$ are $i$ times the spring
stretches $B^{T}y$, and the rest is zero padding. Then
$i\,\dot\psi=H\psi$ with $H=-\begin{pmatrix}0&B\\B^{T}&0\end{pmatrix}$
is exactly $\ddot x=-M^{-1}Fx$. The initial state of §6,
$(|0\rangle-|1\rangle)/\sqrt2$, means $x(0)=0$ and
$\dot x_0=-\dot x_1=1/\sqrt2$: two neighbours pushed apart.

**Decoding.** Velocities are $\mathrm{Re}\,\psi[:N]/\sqrt m$. Positions
come from the stretches, $y=\mathrm{lstsq}(B^{T}, \mathrm{Im}\,\psi[N:N+P])$.
For the chain, which has free ends and no wall springs, $B^{T}$ has a
one-dimensional null space spanned by $\sqrt m$: translating the whole
chain stretches no spring, so the stretches cannot say where the chain
is. The least-squares $y$ is orthogonal to $\sqrt m$, i.e. it has its
centre of mass at the origin, and the missing piece is the centre of
mass itself, which moves uniformly,
$x_\mathrm{cm}(t)=x_\mathrm{cm}(0)+t\,P_\mathrm{tot}/M_\mathrm{tot}$
with $P_\mathrm{tot}=\sum_i m_i\dot x_i$ conserved. `decode_state`
adds $x_\mathrm{cm}(t)\sqrt m$ back (here $P_\mathrm{tot}=0$, so the
centre stays at the origin). With wall springs (the seeded random
network) $B^{T}$ has full column rank and no correction is needed; any
other null space (a disconnected network) raises an error.

The first cell below defines `decode_state` and tests the encoding
alone: decoding the *exact* quantum evolution (no Trotter error) must
reproduce the classical solution to rounding error.

In [ ]:
####################################################
# Decode (x, xdot) from a state, check the encoding
#  - decode_state(psi, B, masses, t, x_cm0=0.0)
#  - classical_solution(F, masses, x0, v0, times)
####################################################

from scipy.linalg import expm

# t_end, the trajectory window, is set in the cost preview (4.1).


def decode_state(psi, B, masses, t, x_cm0=0.0):
    """Decode positions and velocities from the encoded state psi.

    Encoding (Babbush et al. 2023): with y = sqrt(M) x,
        psi = (ydot, i B^T y)     (components 0..N-1, N..N+P-1),
    zero-padded to 2^n. Returns (x, xdot, diagnostics).

    - xdot = Re(psi[:N]) / sqrt(m).
    - b = Im(psi[N:N+P]) = B^T y; y = lstsq(B^T, b).
    - If B^T has a null space it must be 1-dimensional and spanned
      by sqrt(m): translating the whole network leaves every spring
      unchanged (free ends, no wall springs). The least-squares y is
      orthogonal to sqrt(m), so the centre of mass is restored by
      adding x_cm(t) sqrt(m), x_cm(t) = x_cm0 + t P_tot / M_tot
      (P_tot = sum m xdot is conserved: no net external force).
    - Anything else (e.g. a disconnected network) raises.
    """
    psi = np.asarray(psi)
    B = np.asarray(B, dtype=float)
    m = np.asarray(masses, dtype=float)
    n_osc, n_cols = B.shape
    sqrt_m = np.sqrt(m)
    ydot, b = psi[:n_osc], psi[n_osc:n_osc + n_cols]

    # Null space of B^T (a map R^N -> R^P), from the SVD. P >= N, so
    # the reduced SVD already has all N right singular vectors.
    _, s, vt = np.linalg.svd(B.T, full_matrices=False)
    null = vt[s <= 1e-10 * s[0]]
    if len(null) > 1:
        raise ValueError(f"B^T has a {len(null)}-dimensional null "
                         "space: the network is disconnected, so "
                         "positions are not determined")
    if len(null) == 1:
        overlap = abs(null[0] @ sqrt_m) / np.linalg.norm(sqrt_m)
        if overlap < 1 - 1e-8:
            raise ValueError("B^T's null vector is not sqrt(m): "
                             "not a whole-network translation")

    y = np.linalg.lstsq(B.T, b.imag, rcond=1e-10)[0]
    # b must lie in the range of B^T (it is B^T y for some y)
    range_residual = np.max(np.abs(B.T @ y - b.imag))
    p_tot = float(sqrt_m @ ydot.real)  # sum_i m_i xdot_i
    if len(null) == 1:
        y = y + (x_cm0 + t * p_tot / m.sum()) * sqrt_m
    diagnostics = {
        "max_im_velocity": float(np.max(np.abs(ydot.imag))),
        "max_re_b": float(np.max(np.abs(b.real))),
        "max_padding": float(np.max(np.abs(psi[n_osc + n_cols:]),
                                    initial=0.0)),
        "range_residual": float(range_residual),
        "momentum": p_tot,
        "null_dim": len(null),
    }
    return y / sqrt_m, ydot.real / sqrt_m, diagnostics


def classical_solution(F, masses, x0, v0, times):
    """Exact solution of M xddot = -F x: z = (x, xdot),
    z(t) = expm(A t) z(0), A = [[0, I], [-M^-1 F, 0]]."""
    m = np.asarray(masses, dtype=float)
    n = len(m)
    A = np.block([[np.zeros((n, n)), np.eye(n)],
                  [-F / m[:, None], np.zeros((n, n))]])
    z0 = np.concatenate([x0, v0])
    z = np.array([expm(A * t) @ z0 for t in times])
    return z[:, :n], z[:, n:]


# --- Encoding check: exact quantum evolution (no Trotter error)
# decoded, against the exact classical solution. This validates the
# encoding and the decoder independently of the Trotter step. ---
_x0 = np.zeros(N)
_v0 = psi0[:N].real / np.sqrt(masses)
_t_enc = t_end * np.array([0.05, 0.3, 1.0])
_xc, _vc = classical_solution(_F, masses, _x0, _v0, _t_enc)
_err_enc = 0.0
for _k, _t in enumerate(_t_enc):
    _psi_q = expm_multiply(-1j * _t * Hp, psi0)
    _xq, _vq, _diag = decode_state(_psi_q, _B_phys, masses, _t)
    _e = max(np.max(np.abs(_xq - _xc[_k])), np.max(np.abs(_vq - _vc[_k])))
    _err_enc = max(_err_enc, _e)
    print(f"t={_t:8.4f}: max|decoded - classical| = {_e:.2e} "
          f"(null dim {_diag['null_dim']}, "
          f"range residual {_diag['range_residual']:.1e})")
print(f"encoding check, exact quantum evolution at 3 times: "
      f"max error = {_err_enc:.2e}")
if _err_enc > 1e-8:
    raise RuntimeError("decoded exact evolution disagrees with the "
                       "classical solution: encoding/decoder bug")

The trajectory run below evolves the state once, sequentially, over
the whole window with the §3 bound's step size for $t=10.762932$, and
keeps a copy of the state at each of the 41 read-out times (the step
count nearest to each time; the largest rounding is half a step, and
all comparisons use the rounded times). For the default $N=30$ that
is $15{,}493$ steps of $\Delta t\approx6.9\times10^{-4}$ (rounding at
most $3.5\times10^{-4}$), and the **expected runtime is about 3
minutes** (161 to 197 s measured here, 10 to 13 ms per step); for
other $N$ the cost preview (§4.1) printed the step count and the
estimate. One progress line covers the whole run.

In [ ]:
####################################################
# Evolve the state along the whole time window
#  - evolve_trajectory(psi, terms=..., dt=...,
#                      step_marks=...)
####################################################

# n_times (41 read-out times) is set in the cost preview (4.1).
r_traj = required_trotter_repetitions(t_end, target_error)
dt_traj = t_end / r_traj
t_grid = np.linspace(0.0, t_end, n_times)
step_marks = np.rint(t_grid / dt_traj).astype(int)
t_actual = step_marks * dt_traj  # times the states really land on
print(f"window t=0..{t_end}: {r_traj} steps of dt={dt_traj:.3e}; "
      f"{n_times} read-out times, largest time rounding "
      f"{np.max(np.abs(t_actual - t_grid)):.2e} (half a step is "
      f"{dt_traj / 2:.2e})")


def evolve_trajectory(psi, *, terms, dt, step_marks, order=2,
                      backend="c", label="trajectory",
                      report_every_s=2.0):
    """Advance psi by step_marks[-1] steps of size dt, returning a
    copy of the state after each of the cumulative step counts in
    step_marks (ascending, may start at 0). One progress line for the
    whole run: step k/n, %, which time point, elapsed, ETA.

    Runs in chunks of about one second of work, like
    evolve_with_progress; consecutive trotterkit.evolve calls of the
    same dt compose, so the states are those of a single long run.
    """
    marks = [int(k) for k in step_marks]
    n_total = marks[-1]
    psi = np.array(psi, dtype=np.complex128)
    states = []
    start = last_report = time.perf_counter()
    done, chunk = 0, 10
    for idx, mark in enumerate(marks):
        while done < mark:
            step = min(chunk, mark - done)
            psi = trotterkit.evolve(psi, t=step * dt, steps=step,
                                    terms=terms, order=order,
                                    backend=backend)
            done += step
            now = time.perf_counter()
            rate = done / (now - start)
            chunk = max(1, int(rate))
            if now - last_report >= report_every_s:
                print(f"{label}: step {done}/{n_total} "
                      f"({100.0 * done / n_total:.0f}%) | time point "
                      f"{idx + 1}/{len(marks)} | elapsed "
                      f"{_mmss(now - start)} | ETA "
                      f"{_mmss((n_total - done) / rate)}   ",
                      end="\r", flush=True)
                last_report = now
        states.append(np.array(psi))
    print(f"{label}: {n_total} steps, {len(marks)} states in "
          f"{_mmss(time.perf_counter() - start)}" + " " * 40)
    return states


_t0 = time.perf_counter()
psi_traj = evolve_trajectory(
    psi0, terms=(x_terms, z_terms, coeff_terms), dt=dt_traj,
    step_marks=step_marks, label=f"N={N} trajectory")
print(f"trajectory evolution: {time.perf_counter() - _t0:.1f}s")

The decoded trajectories are compared with the exact classical
solution at the same times. The decoder's diagnostics should be
rounding-level: $\mathrm{Im}\,\dot y$, $\mathrm{Re}\,B^{T}y$ and the
padding components are zero for the exact state, and $\mathrm{Im}\,\psi$
of the stretch block must lie in the range of $B^{T}$. Total momentum
is conserved. Finally the state-level error is checked against the
$0.05$ target the §3 bound was built for.

In [ ]:
####################################################
# Validate the decoded trajectories
#  - against the exact classical solution
#  - state-level error against the Trotter target
####################################################

_x_ref, _v_ref = classical_solution(_F, masses, _x0, _v0, t_actual)

x_traj = np.empty((n_times, N))
v_traj = np.empty((n_times, N))
_diag_max = {"max_im_velocity": 0.0, "max_re_b": 0.0,
             "max_padding": 0.0, "range_residual": 0.0}
_momentum = np.empty(n_times)
_null_dim = 0
for _k in range(n_times):
    x_traj[_k], v_traj[_k], _d = decode_state(
        psi_traj[_k], _B_phys, masses, t_actual[_k])
    _momentum[_k] = _d["momentum"]
    _null_dim = _d["null_dim"]
    for _name in _diag_max:
        _diag_max[_name] = max(_diag_max[_name], _d[_name])

_x_err = np.max(np.abs(x_traj - _x_ref))
_v_err = np.max(np.abs(v_traj - _v_ref))
_p_drift = np.max(np.abs(_momentum - _momentum[0]))
print(f"vs exact classical solution, {n_times} times x {N} "
      f"oscillators:\n  max |x error|    = {_x_err:.2e}   "
      f"(max |x| = {np.max(np.abs(_x_ref)):.3f})\n"
      f"  max |xdot error| = {_v_err:.2e}   "
      f"(max |xdot| = {np.max(np.abs(_v_ref)):.3f})")
print("decoder diagnostics (maxima over all times): "
      f"max|Im xdot part| = {_diag_max['max_im_velocity']:.1e}, "
      f"max|Re of B^T y part| = {_diag_max['max_re_b']:.1e}, "
      f"max|padding| = {_diag_max['max_padding']:.1e}, "
      f"range residual = {_diag_max['range_residual']:.1e}")
_p_note = ("conserved: free ends, no external force" if _null_dim
           else "not conserved: wall springs push on the network")
print(f"total momentum ({_p_note}): initial {_momentum[0]:.2e}, max "
      f"drift {_p_drift:.2e}")

# State-level error at every read-out time: the Trotterized state
# against exact evolution (dense eigendecomposition of the dim x dim
# H, done once) - this is the quantity the section 3 bound controls.
_t0 = time.perf_counter()
_E, _V = np.linalg.eigh(Hp.toarray())
_coef = _V.conj().T @ psi0
_state_err = max(
    np.linalg.norm(psi_traj[_k]
                   - _V @ (np.exp(-1j * _E * t_actual[_k]) * _coef))
    for _k in range(n_times))
print(f"state error max_k ||psi_trotter - psi_exact|| = "
      f"{_state_err:.2e} (target {target_error}; "
      f"{time.perf_counter() - _t0:.1f}s)")
if _state_err > target_error:
    raise RuntimeError("Trotterized trajectory exceeds the Trotter "
                       "error target")

In [ ]:
####################################################
# Plot the read-out trajectories
#  - positions: Trotter read-out (markers) over exact
#  - space-time map of all N oscillators
#  - velocities
####################################################

import matplotlib.pyplot as plt

_t_fine = np.linspace(0.0, t_actual[-1], 400)
_x_fine, _v_fine = classical_solution(_F, masses, _x0, _v0, _t_fine)
# Up to five oscillators: the kicked pair (0, 1), its neighbour, and
# two further down the chain. The disturbance spreads about one site
# per unit time on the unit chain, so sites up to about t_end are
# reached within the window; indices >= N are dropped for small N.
_shown = [i for i in (0, 1, 2, 5, 10) if i < N]
_title_tail = (f"N={N} chain, {r_traj} Trotter steps "
               f"(2nd order)")

fig_pos, ax = plt.subplots(figsize=(10, 4.8))
for _i, _c in zip(_shown, plt.cm.tab10.colors):
    ax.plot(_t_fine, _x_fine[:, _i], color=_c, lw=1.2)
    ax.plot(t_actual, x_traj[:, _i], "o", color=_c, ms=4.5,
            label=f"oscillator {_i}")
ax.set_xlabel("time")
ax.set_ylabel("position $x_i$")
ax.set_title("Positions read out from the Trotterized state "
             "(markers)\nover the exact classical solution (lines), "
             + _title_tail)
ax.legend(ncol=5, fontsize=8, loc="upper center",
          bbox_to_anchor=(0.5, -0.15))
ax.grid(alpha=0.3)
fig_pos.tight_layout()
plt.show()

fig_st, ax = plt.subplots(figsize=(10, 5))
_lim = np.max(np.abs(x_traj))
_mesh = ax.pcolormesh(t_actual, np.arange(N), x_traj.T,
                      cmap="RdBu_r", vmin=-_lim, vmax=_lim,
                      shading="nearest")
fig_st.colorbar(_mesh, ax=ax, label="position $x_i$")
ax.set_xlabel("time")
ax.set_ylabel("oscillator index $i$")
ax.set_title("Space-time map of the read-out positions, "
             + _title_tail)
fig_st.tight_layout()
plt.show()

fig_vel, ax = plt.subplots(figsize=(10, 4.8))
for _i, _c in zip(_shown, plt.cm.tab10.colors):
    ax.plot(_t_fine, _v_fine[:, _i], color=_c, lw=1.2)
    ax.plot(t_actual, v_traj[:, _i], "o", color=_c, ms=4.5,
            label=f"oscillator {_i}")
ax.set_xlabel("time")
ax.set_ylabel(r"velocity $\dot{x}_i$")
ax.set_title("Velocities read out from the Trotterized state "
             "(markers)\nover the exact classical solution (lines), "
             + _title_tail)
ax.legend(ncol=5, fontsize=8, loc="upper center",
          bbox_to_anchor=(0.5, -0.15))
ax.grid(alpha=0.3)
fig_vel.tight_layout()
plt.show()

## 8. Signs and phases without Classiq: our own Hadamard test

§7 read the trajectories from the full statevector, which a classical
simulator can print but a quantum device cannot: a device returns only
measurement outcomes. N=2 §6.7 recovers amplitudes, with their signs
and phases, by a Hadamard test run through Classiq. This section builds
the same test from the pieces already verified here - `trotterkit` for
the circuit's action, NumPy for the shot noise, `decode_state` for the
read-out - with no Classiq anywhere.

The Hadamard test for estimating $\mathrm{Re}/\mathrm{Im}$ of an amplitude is used as described in D. Aharonov, V. Jones, and Z. Landau, “A Polynomial Quantum Algorithm for Approximating the Jones Polynomial,” *Proc. 38th ACM Symp. on Theory of Computing (STOC)*, 2006, pp. 427–436 (arXiv:quant-ph/0511096).

**Goal.** Estimate $a_k(t)=\langle k|\psi(t)\rangle=\langle k|U|\psi_0\rangle$,
$U$ the Trotterized evolution, from measurement outcomes alone.

**Circuit.** One ancilla qubit $a$ is the *most* significant bit, basis
index $a\cdot2^n+j$ with $n$ system qubits ($n=9$ for the default
$N=30$).

1. $H$ on $a$, then prepare $|k\rangle$ on the system controlled on
   $a=0$ and $|\psi_0\rangle$ controlled on $a=1$:
   $\tfrac1{\sqrt2}\big(|0\rangle_a|k\rangle+|1\rangle_a|\psi_0\rangle\big)$.
2. Apply $U$ controlled on $a=1$:
   $\tfrac1{\sqrt2}\big(|0\rangle|k\rangle+|1\rangle|\phi\rangle\big)$,
   $\phi=U\psi_0$.
3. Optionally $S^\dagger=\mathrm{diag}(1,-i)$ on $a$, then $H$ on $a$,
   then measure $a$.

**Derivation.** Without $S^\dagger$ the final $H$ sends the $a=0$
branch to $\tfrac12(|k\rangle+|\phi\rangle)$, so
$$P(a{=}0)=\tfrac14\big\|\,|k\rangle+|\phi\rangle\big\|^2
=\tfrac14\big(2+2\,\mathrm{Re}\langle k|\phi\rangle\big)
=\tfrac12\big(1+\mathrm{Re}\,a_k\big).$$
With $S^\dagger$ first, the $a=1$ branch is multiplied by $-i$ and the
$a=0$ branch becomes $\tfrac12(|k\rangle-i|\phi\rangle)$, so
$P(a{=}0)=\tfrac12\big(1+\mathrm{Re}(-i\,a_k)\big)=\tfrac12\big(1+
\mathrm{Im}\,a_k\big)$, since $\mathrm{Re}(-iz)=\mathrm{Im}\,z$.
Two circuits per amplitude give $\mathrm{Re}\,a_k$ and
$\mathrm{Im}\,a_k$, with their signs.

**Controlled evolution with `trotterkit`.** For a term $cP$ with $P$ on
the $n$ system qubits, controlled on $a=1$, using
$|1\rangle\langle1|=(I-Z_a)/2$:
$$e^{-i\theta c\,P\otimes|1\rangle\langle1|}
=e^{-i\theta\,(c/2)\,P}\;e^{+i\theta\,(c/2)\,Z_aP},$$
two commuting factors. In `trotterkit`'s convention
($P=i^{\mathrm{popcount}(x\&z)}X^xZ^z$) the second factor is the term
$(x,\,z\,|\,2^n)$ with coefficient $-c/2$: $Z_a$ adds a $z$ bit and no
$x$ bit, so the phase prefactor is unchanged. Replacing every term by
the adjacent pair $[(x,z,c/2),\,(x,z|2^n,-c/2)]$ and running the
ordinary order-2 formula on $n+1$ qubits therefore gives *exactly* the
controlled version of the original Strang step: the pair commutes, so
the forward/reverse order inside a pair is irrelevant, the $a=0$ branch
sees $e^{-i\theta c P/2}e^{+i\theta cP/2}=I$ and the $a=1$ branch
($Z_a=-1$) sees $e^{-i\theta cP}$. The cell below builds this list and
checks both branches, and both measurement probabilities, against the
uncontrolled $n$-qubit evolution.

In [ ]:
####################################################
# Controlled evolution via doubled terms, verified
#  - a=0 branch untouched, a=1 branch = U psi0
#  - P(a=0) after the final H (and S^dagger)
####################################################

_t_hc = time.perf_counter()
# Pair (x, z, c/2), (x, z | bit_a, -c/2) for every original term,
# ancilla = MSB (controlled_terms, defined in the cost preview 4.1).
x_ctl, z_ctl, coeff_ctl = controlled_terms(x_terms, z_terms,
                                           coeff_terms, n_qubits)
terms_ctl = (x_ctl, z_ctl, coeff_ctl)
dim_ctl = 2 * dim

_t_hc_check = 0.5
_r_hc = required_trotter_repetitions(_t_hc_check, target_error)
_dt_hc = _t_hc_check / _r_hc
# Reference: the n-qubit evolution of psi0 with the same steps.
_phi_ref = evolve_with_progress(
    psi0, terms=(x_terms, z_terms, coeff_terms), dt=_dt_hc, n_steps=_r_hc,
    label=f"{n_qubits}-qubit reference")

_worst = {"branch0": 0.0, "branch1": 0.0, "P0_re": 0.0, "P0_im": 0.0}
for _k_hc in (0, 2):  # a basis state psi0 overlaps, and one it does not
    _ket_k = np.zeros(dim, dtype=np.complex128)
    _ket_k[_k_hc] = 1.0
    _psi_in = np.concatenate([_ket_k, psi0]) / np.sqrt(2)  # a = MSB
    _psi_out = evolve_with_progress(
        _psi_in, terms=terms_ctl, dt=_dt_hc, n_steps=_r_hc,
        label=f"controlled, k={_k_hc}")
    _h0, _h1 = _psi_out[:dim], _psi_out[dim:]
    _worst["branch0"] = max(_worst["branch0"], np.max(
        np.abs(_h0 - _ket_k / np.sqrt(2))))
    _worst["branch1"] = max(_worst["branch1"], np.max(
        np.abs(_h1 - _phi_ref / np.sqrt(2))))
    # final H on a: the a=0 branch is (h0 + h1)/sqrt2; with S^dagger
    # first, h1 is multiplied by -i.
    _p0_re = np.linalg.norm((_h0 + _h1) / np.sqrt(2)) ** 2
    _p0_im = np.linalg.norm((_h0 - 1j * _h1) / np.sqrt(2)) ** 2
    _a_k = _ket_k.conj() @ _phi_ref
    _worst["P0_re"] = max(_worst["P0_re"],
                          abs(_p0_re - (1 + _a_k.real) / 2))
    _worst["P0_im"] = max(_worst["P0_im"],
                          abs(_p0_im - (1 + _a_k.imag) / 2))
_n_rot = _r_hc * 2 * len(x_ctl)  # rotations per run (Strang: 2x)
print(f"controlled evolution at t={_t_hc_check}, {_r_hc} steps, "
      f"{len(x_ctl)} doubled terms on {n_qubits + 1} qubits "
      f"({_n_rot:.2e} rotations per run):\n"
      f"  a=0 branch vs |k>/sqrt2:        {_worst['branch0']:.2e}\n"
      f"  a=1 branch vs {n_qubits}-qubit psi/sqrt2: "
      f"{_worst['branch1']:.2e}\n"
      f"  P(a=0) vs (1+Re a_k)/2:          {_worst['P0_re']:.2e}\n"
      f"  P(a=0) vs (1+Im a_k)/2:          {_worst['P0_im']:.2e}\n"
      f"  (final norm drift {abs(np.linalg.norm(_psi_out) - 1):.1e}; "
      f"{time.perf_counter() - _t_hc:.1f}s)")
# Agreement is limited by rounding, not by the construction: each
# Pauli rotation perturbs the state by ~1e-16 and these add up (at
# N=30, ~1e7 rotations per run and a norm drift of ~5e-10; the 5-step
# runs agree to ~1e-12). 1e-8 is far below anything the read-out can
# resolve. The rotation count grows with N (steps x terms), so above
# 1e7 rotations the tolerance grows in proportion - the same margin
# per rotation as at N=30, never looser than there.
_tol_hc = 1e-8 * max(1.0, _n_rot / 1e7)
if max(_worst.values()) > _tol_hc:
    raise RuntimeError("doubled-term controlled evolution is not the "
                       "controlled Strang step")

### 8.1 Emulated measurement with shot noise

Because the controlled $U$ leaves the $a=0$ branch alone, every
amplitude at time $t$ comes from the *same* $\phi=U\psi_0$ that §7
already computed at its 41 times; no new evolution is needed. For
each needed amplitude the exact outcome probabilities are
$p_\mathrm{re}=(1+\mathrm{Re}\,a)/2$ and
$p_\mathrm{im}=(1+\mathrm{Im}\,a)/2$. Each circuit is run $s$ times
(a binomial draw from a seeded NumPy `Generator`), the estimate is
$\hat a=(2\hat p_\mathrm{re}-1)+i(2\hat p_\mathrm{im}-1)$, and an
estimated state is rebuilt and handed to `decode_state`.

Only the $N$ velocity components and the stretches of the nonzero
springs ($N-1$ for the chain) are needed, $2N-1$ amplitudes in all.
The other components (the zero-spring columns of $B$ and the padding)
are zero by construction and are never measured. Measuring the
$2N-1$ amplitudes at $41$ times, two circuits each, is
$41\times(2N-1)\times2$ circuits. For the default $N=30$ that is
$59$ of the $512$ components (the other $436$ zero-spring columns and
$17$ padding components are skipped) and $4{,}838$ circuits.

The cell draws one experiment at $4000$ shots per circuit, with error
bars from the spread of $100$ repeated experiments, and plots the
decoded positions over the exact classical solution.

In [ ]:
####################################################
# Emulated Hadamard-test measurement and decoding
#  - estimate_state(psi, shots, rng)
#  - decoded positions, 4000 shots per circuit
####################################################

amp_idx = _keep  # N velocity + nonzero-spring components
n_amp = len(amp_idx)
n_circuits = n_times * n_amp * 2
print(f"{n_amp} amplitudes x {n_times} times x 2 circuits "
      f"= {n_circuits} circuits (the other {dim - n_amp} "
      f"components are zero by construction)")


def estimate_state(psi, shots, rng):
    """Emulate the two Hadamard-test circuits per amplitude: draw
    binomial counts of the ancilla outcome 0 from the exact
    probabilities, estimate Re a and Im a, and rebuild a state that
    is zero off amp_idx."""
    a = psi[amp_idx]
    p_re = (1.0 + a.real) / 2.0
    p_im = (1.0 + a.imag) / 2.0
    re_hat = 2.0 * rng.binomial(shots, np.clip(p_re, 0, 1)) / shots - 1.0
    im_hat = 2.0 * rng.binomial(shots, np.clip(p_im, 0, 1)) / shots - 1.0
    est = np.zeros(dim, dtype=np.complex128)
    est[amp_idx] = re_hat + 1j * im_hat
    return est


def decode_estimates(shots, rng):
    """One full experiment: every time, every amplitude. Returns
    decoded positions (n_times, N)."""
    xs = np.empty((n_times, N))
    for k in range(n_times):
        xs[k], _, _ = decode_state(estimate_state(psi_traj[k], shots,
                                                  rng),
                                   _B_phys, masses, t_actual[k])
    return xs


_t_hm = time.perf_counter()
shots_demo = 4000
rng_demo = np.random.default_rng(2026)
x_hat = decode_estimates(shots_demo, rng_demo)
x_reps = np.array([decode_estimates(shots_demo, rng_demo)
                   for _ in range(100)])
x_std = x_reps.std(axis=0)  # std of the estimate, per time and site
_err_demo = np.max(np.abs(x_hat - _x_ref))
print(f"{shots_demo} shots per circuit: max |x_hat - x_exact| = "
      f"{_err_demo:.3e} (one experiment); rms over 100 repeats "
      f"{np.sqrt(np.mean((x_reps - _x_ref) ** 2)):.3e}; "
      f"max std of the estimate {x_std.max():.3e}; "
      f"max |x| = {np.max(np.abs(_x_ref)):.3f} "
      f"({time.perf_counter() - _t_hm:.1f}s)")

fig_hadamard, ax = plt.subplots(figsize=(10, 4.8))
for _i, _c in zip(_shown, plt.cm.tab10.colors):
    ax.plot(_t_fine, _x_fine[:, _i], color=_c, lw=1.2)
    ax.errorbar(t_actual, x_hat[:, _i], yerr=x_std[:, _i], fmt="o",
                color=_c, ms=4, capsize=2, lw=1,
                label=f"oscillator {_i}")
ax.set_xlabel("time")
ax.set_ylabel("position $x_i$")
ax.set_title(f"Positions from the emulated Hadamard test, "
             f"{shots_demo} shots per circuit\n"
             "(markers, error bars = std of the estimate) over the "
             "exact classical solution (lines)")
ax.legend(ncol=5, fontsize=8, loc="upper center",
          bbox_to_anchor=(0.5, -0.15))
ax.grid(alpha=0.3)
fig_hadamard.tight_layout()
plt.show()

The error of an estimate from $s$ shots scales as $1/\sqrt s$: each
$\hat p$ has standard deviation $\sqrt{p(1-p)/s}\le1/(2\sqrt s)$, and
the decoder is linear. Positions are sums of many stretches, so the
noise accumulates along the chain: for the default $N=30$, at $4000$
shots the maximum error is about $0.1$ against $|x|\le0.5$. The next
cell measures this: the maximum position error over all times and
oscillators, measured from the noise-free read-out of §7 so that the
Trotter error (the same at every shot count, and not negligible at
small $N$, where the §3 bound allows few, large steps) does not mask
the shot noise, against shots per
circuit from $10^2$ to $10^6$, for several seeds, with the fitted
power law and a $1/\sqrt s$ reference. It then turns the fit into a
shot budget for a target accuracy.

In [ ]:
####################################################
# Shot scaling of the decoded positions
#  - max position error vs shots per circuit
#  - shot budget for a target accuracy
####################################################

_t_sc = time.perf_counter()
shot_grid = np.array([10 ** (2 + 0.5 * i) for i in range(9)]).astype(int)
n_seeds = 8
_errs = np.empty((len(shot_grid), n_seeds))
_n_runs = len(shot_grid) * n_seeds
_last = _t_sc
for _i_s, _s in enumerate(shot_grid):
    for _seed in range(n_seeds):
        _xs = decode_estimates(int(_s), np.random.default_rng(
            1000 * _i_s + _seed))
        # Shot error alone: against the noise-free read-out x_traj of
        # section 7, the mean of the estimate. Against the exact
        # solution, the Trotter error (the same at every shot count)
        # would flatten the slope wherever it is not negligible - at
        # N=30 it is 2e-7, but at N=2 it is ~2e-2, as large as the
        # shot error at 1e6 shots.
        _errs[_i_s, _seed] = np.max(np.abs(_xs - x_traj))
        _done = _i_s * n_seeds + _seed + 1
        _now = time.perf_counter()
        if _now - _last >= 2.0 or _done == _n_runs:
            _rate = _done / (_now - _t_sc)
            print(f"shot scaling: run {_done}/{_n_runs} "
                  f"({int(_s):,} shots per circuit) | elapsed "
                  f"{_mmss(_now - _t_sc)} | ETA "
                  f"{_mmss((_n_runs - _done) / _rate)}   ",
                  end="\r", flush=True)
            _last = _now
print(" " * 79, end="\r")
_mean_err = _errs.mean(axis=1)
_slope, _icpt = np.polyfit(np.log10(shot_grid), np.log10(_mean_err), 1)
# Best C in error = C / sqrt(s): the mean of log10(err * sqrt(s)).
_c_fit = 10 ** np.mean(np.log10(_mean_err)
                       + 0.5 * np.log10(shot_grid))
print(f"log-log slope of max position error vs shots: {_slope:.3f} "
      f"(1/sqrt(s) would be -0.500); error ~ {_c_fit:.2f}/sqrt(s) "
      f"({time.perf_counter() - _t_sc:.1f}s)")
print("shots per circuit -> mean max error: " + ", ".join(
    f"{int(_s):,}: {_e:.2e}" for _s, _e in zip(shot_grid, _mean_err)))
if not -0.6 < _slope < -0.4:
    raise RuntimeError("shot noise does not scale as 1/sqrt(shots)")

fig_shots, ax = plt.subplots(figsize=(7, 4.8))
for _j in range(n_seeds):
    ax.loglog(shot_grid, _errs[:, _j], "o", color="C0", alpha=0.3,
              ms=4)
ax.loglog(shot_grid, _mean_err, "o-", color="C0",
          label=f"mean over {n_seeds} seeds (slope {_slope:.2f})")
ax.loglog(shot_grid, _c_fit / np.sqrt(shot_grid), "k--",
          label=r"$C/\sqrt{s}$ reference")
ax.set_xlabel("shots per circuit $s$")
ax.set_ylabel("max position error")
ax.set_title("Hadamard-test position error vs shots")
ax.legend()
ax.grid(alpha=0.3, which="both")
fig_shots.tight_layout()
plt.show()

print(f"\ncircuits for the full trajectory: {n_times} times x {n_amp} "
      f"amplitudes x 2 = {n_circuits}")
# Shot noise only: the Trotter error of section 7 adds to this.
for _eps in (1e-2, 1e-3):
    _s_need = math.ceil((_c_fit / _eps) ** 2)
    print(f"target max position error {_eps:g}: about {_s_need:,} "
          f"shots per circuit, {n_circuits * _s_need:.2e} shots in "
          f"total")

### 8.2 Exact circuit size from our own gate list

The statevector checks above say the circuit is *right*; they do not
say what it *costs*. The count below uses our own gate model and the
actual Pauli masks, not an estimate from formulas.

- An uncontrolled rotation $e^{-i\theta P}$ of a string of weight
  $w=\mathrm{popcount}(x|z)$ is: a basis change on every $X$ qubit
  ($H$ before and after, 2 single-qubit Cliffords) and every $Y$ qubit
  ($S^\dagger H$ before, $HS$ after, 4 Cliffords), a CNOT ladder of
  $2(w-1)$ CNOTs, and one $R_z$.
- The controlled rotation needs the same basis changes and ladder
  (they cancel when the rotation is off, so they need no control) but
  the $R_z$ becomes a controlled-$R_z$: 2 CNOTs and 2 $R_z$. A weight-0
  term is a global phase, which under control is one phase gate on the
  ancilla.
- A Strang step applies each term twice (forward, then reverse; the
  merge of the two middle terms is ignored, a negligible saving).
- Per circuit, on top of the $r$ steps: $H$ on the ancilla at the
  start and end (plus $S^\dagger$ for the $\mathrm{Im}$ circuits);
  $|k\rangle$ controlled on $a=0$ is $X_a$, one CNOT per set bit of
  $k$, $X_a$; $|\psi_0\rangle=(|0\rangle-|1\rangle)/\sqrt2$
  controlled on $a=1$ is a CNOT plus a controlled-$H$ (1 CNOT and 2
  rotations) on the least significant qubit.
- A trajectory re-runs $U$ from scratch for every read-out time: the
  device cannot keep a copy of its state, so the 41 times cost
  $\sum_j r_j$ steps per amplitude, not $r(t_\mathrm{end})$.

In [ ]:
####################################################
# Exact gate counts from the Pauli masks
#  - gate_counts(x, z)
#  - per-step, per-circuit and whole-trajectory totals
####################################################

from IPython.display import Markdown, display


def gate_counts(x, z):
    """Per-term gate counts from the masks (our gate model).
    Returns dict of totals over all terms for ONE application of
    every term: uncontrolled CNOT / Clifford / Rz, and the same for
    the controlled version."""
    x = np.asarray(x, dtype=np.uint64)
    z = np.asarray(z, dtype=np.uint64)
    w = np.bitwise_count(x | z).astype(np.int64)
    n_y = np.bitwise_count(x & z).astype(np.int64)
    n_x = np.bitwise_count(x).astype(np.int64) - n_y
    cliff = 2 * n_x + 4 * n_y
    ladder = np.maximum(2 * (w - 1), 0)
    live = w > 0
    return {
        "terms": len(w),
        "u_cnot": int(ladder.sum()),
        "u_cliff": int(cliff.sum()),
        "u_rz": int(live.sum()),
        # controlled: Rz -> CRz (2 CNOT + 2 Rz); weight 0 -> 1 phase
        "c_cnot": int((ladder + 2 * live).sum()),
        "c_cliff": int(cliff.sum()),
        "c_rz": int((2 * live + (~live)).sum()),
    }


def step_counts(gc):
    """Per controlled Strang step: every term applied twice."""
    return {k: 2 * gc[k] for k in ("c_cnot", "c_cliff", "c_rz")}


def prep_counts(k, with_sdg):
    """Per-circuit gates besides the r steps, for amplitude index k:
    CNOT / single-qubit."""
    cnot = bin(int(k)).count("1") + 2  # |k> bits; CNOT + CH for psi0
    single = 2 + 2 + 2 + (1 if with_sdg else 0)  # H,H ; X,X ; CH rots
    return cnot, single


_gc_full = gate_counts(x_terms, z_terms)
_gc_pruned = gate_counts(_xp, _zp)
_sc_full, _sc_pruned = step_counts(_gc_full), step_counts(_gc_pruned)


def circuit_total(sc, steps, k, with_sdg):
    pc, ps = prep_counts(k, with_sdg)
    return (steps * sc["c_cnot"] + pc,
            steps * (sc["c_cliff"] + sc["c_rz"]) + ps)


_rows = []
for _name, _gc, _sc in ((f"{n_qubits} qubits (full layout)",
                         _gc_full, _sc_full),
                        (f"{_n_q_pruned} qubits (pruned, §2.1)",
                         _gc_pruned, _sc_pruned)):
    _rows.append((_name + ", one uncontrolled step",
                  2 * _gc["u_cnot"],
                  2 * (_gc["u_cliff"] + _gc["u_rz"])))
    _rows.append((_name + ", one controlled step",
                  _sc["c_cnot"], _sc["c_cliff"] + _sc["c_rz"]))
_mean_pop = np.mean([bin(int(k)).count("1") for k in amp_idx])
for _t_c in (1.0, t_end):
    _r_c = required_trotter_repetitions(_t_c, target_error)
    _cn, _s1 = circuit_total(_sc_full, _r_c, amp_idx[0], False)
    # popcount(k) averaged over the read-out amplitudes
    _cn += round(_mean_pop) - bin(int(amp_idx[0])).count("1")
    _rows.append((f"{n_qubits} qubits, one circuit at t={_t_c:g} "
                  f"({_r_c:,} steps)", _cn, _s1))

# Whole trajectory: n_times x n_amp amplitudes x {Re, Im} circuits.
_tot_cnot = _tot_single = 0
for _m in step_marks:
    for _k in amp_idx:
        for _sdg in (False, True):
            _c, _s = circuit_total(_sc_full, int(_m), _k, _sdg)
            _tot_cnot += _c
            _tot_single += _s
_rows.append((f"{n_qubits} qubits, whole trajectory "
               f"({n_circuits:,} circuits)",
              _tot_cnot, _tot_single))

_tab = ["| quantity | CNOTs | single-qubit gates (Cliffords + Rz) |",
        "|---|---:|---:|"]
_tab += [f"| {n} | {c:,} | {s:,} |" for n, c, s in _rows]
display(Markdown("\n".join(_tab)))
print(f"terms: full {_gc_full['terms']:,}, pruned "
      f"{_gc_pruned['terms']:,}; qubits incl. ancilla: "
      f"{n_qubits + 1} and {_n_q_pruned + 1}")
print(f"full layout, per controlled step: {_sc_full['c_cnot']:,} CNOTs, "
      f"{_sc_full['c_cliff']:,} Cliffords, {_sc_full['c_rz']:,} Rz")
print(f"mean popcount of the {n_amp} read-out indices: "
      f"{_mean_pop:.2f}")

### 8.3 What the table says

For the default $N=30$, a single controlled Strang step of the full
layout costs about $4\times10^5$ CNOTs, and a circuit needs thousands
of steps, so one circuit at $t=10.76$ is billions of CNOTs and the
whole trajectory is thousands of times beyond that; both the per-step
cost (more and longer Pauli strings) and the step count grow with
$N$. This is the cost of the loose §3 bound on a dense, non-local
Pauli decomposition; it is why this notebook evolves the state
classically and why the test above is an emulation. The pruned chain
(6 qubits at $N=30$) is about 70 times cheaper per step there, but its
Pauli terms are still long strings, so even there hardware cost, not
shot noise, is the bottleneck.

## 9. Next steps

This notebook stops at a verified Hamiltonian, a verified Pauli
decomposition, a verified (if loose) Trotter-repetition bound, a
verified Trotterized evolution for any $N\ge2$ whose positions and
velocities are read out and checked against the exact classical
solution (§7), and an emulated Hadamard test with an exact circuit-size
count (§8) - the pieces that do not scale past roughly $N=4$ in the
original symbolic/Classiq approach. **Not yet done, deliberately
deferred to the next pass:**

- Decoding positions and velocities and plotting them against the
  classical solution is done: see §7.
- Generalizing N=2 §6.7's Hadamard-test signed-trajectory recovery to
  the chosen $N$ ($9$ system qubits at $N=30$) is done, without
  Classiq: see §8 (controlled
  evolution verified with `trotterkit`, emulated shot noise, decoded
  trajectories, exact gate counts).
- Performance work is done for this notebook: `trotterkit` provides
  threaded C, C++ and Rust backends, compared by measurement, with C
  as the default.
- A tighter, exact Trotter bound at this scale (the 4th-order-specific
  Proposition J.2 bound used in N=2 §6.6.1 needs a 3-way commuting
  partition of the Hamiltonian's terms, verified by hand for $N=2$'s
  12 terms - finding and verifying an analogous partition for
  thousands of terms is its own piece of work, not attempted here).
- Hardware cost: for the default $N=30$, §8.2 counts about
  $4\times10^5$ CNOTs per controlled Strang step and billions per
  circuit, so running this on a device
  needs a much shorter circuit - a tighter step count than the §3
  bound, fewer or more local Pauli terms, or a different product
  formula - not just more shots.

## Summary

- `paulikit` replaces SymPy and the brute-force decomposition for
  any $N\ge2$, set in one parameter cell; at the default $N=30$ (9
  qubits) the nearest-neighbor chain has $14{,}080$ nonzero Pauli
  terms, each checked by direct projection plus a
  Parseval completeness check, and the Hamiltonian itself is checked
  against Newton's equations.
- A coefficient-only bound (using which Pauli strings anticommute)
  replaces the per-term SVDs of the exact Trotter bound; it is
  looser, so it can only over-ask for repetitions, never under-ask.
  It is about 2x tighter in alpha than the plain closed form at
  $N=30$, and checked against the exact bound at $N=2$.
- `trotterkit` applies each Pauli exponential without building a
  matrix, so the $t=1.0$ evolution runs in seconds at $N=30$, and a
  cost preview estimates the run time at any $N$ before the long
  cells; the evolution is verified against
  SciPy's exact `expm_multiply`, with a progress tracker on the
  long loops.
- The positions and velocities are decoded from the evolved state
  (free-end zero mode handled by the centre-of-mass correction) at 41
  times over $t\in[0, 10.762932]$ and agree with the exact classical
  solution; the state-level error stays below the $0.05$ target.
- Our own Hadamard test (§8) recovers signs and phases from emulated
  measurements: the controlled evolution is an exactly doubled term
  list run on $n+1$ qubits (10 at $N=30$), checked against the plain evolution; the
  decoded positions converge as $1/\sqrt{\text{shots}}$; and the
  gate list gives the exact cost, about $4\times10^5$ CNOTs per
  controlled step at $N=30$.

## Further Reading

- A. M. Childs, Y. Su, M. C. Tran, N. Wiebe, S. Zhu, "Theory of Trotter
  Error with Commutator Scaling", Phys. Rev. X 11, 011020 (2021),
  [arXiv:1912.08854](https://arxiv.org/abs/1912.08854)
- R. Babbush et al., "Exponential Quantum Speedup in Simulating
  Coupled Classical Oscillators", Phys. Rev. X 13, 041041 (2023),
  [arXiv:2303.13012](https://arxiv.org/abs/2303.13012)
- `paulikit`: [PyPI](https://pypi.org/project/paulikit/),
  [arXiv:2610.08299](https://arxiv.org/abs/2610.08299)
- Related notebook:
  [`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb)